In [1]:
! source ../../../../devel/setup.bash

In [2]:
# import motoman_config
import sys
sys.path.insert(0,'/usr/local/lib/python3.11/site-packages')

import pinocchio as pin
import hppfcl
from pinocchio.visualize import MeshcatVisualizer

In [3]:
# motoman = motoman_config.MotomanSDA10F()
robot_urdf = "../../../motoman/motoman_sda10f_moveit_config/config/gazebo_motoman_sda10f.urdf"
package_dirs = []
package_dirs.append("../../../motoman")
package_dirs.append("../../../motoman/robotiq")

model, collision_model, visual_model = pin.buildModelsFromUrdf(robot_urdf, package_dirs=package_dirs)

# Start a new MeshCat server and client.
# Note: the server can also be started separately using the "meshcat-server" command in a terminal:
# this enables the server to remain active after the current script ends.
#
# Option open=True pens the visualizer.
# Note: the visualizer can also be opened seperately by visiting the provided URL.    
viz = MeshcatVisualizer(model, collision_model, visual_model)
viz.initViewer(open=True)

# Load the robot in the viewer.
viz.loadViewerModel()

# Display a robot configuration.
q0 = pin.neutral(model)
print('q0: ')
print(q0)
viz.display(q0)


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7055/static/
q0: 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [4]:
import os
os.getcwd()

'/home/kchen/Robotics/workspace/src/lab_vbnpm/playgrounds/collision_checker'

In [5]:
# motoman = motoman_config.MotomanSDA10F()
robot_urdf = "../../../motoman/motoman_sda10f_moveit_config/config/gazebo_motoman_sda10f.urdf"
package_dirs = []
package_dirs.append("../../../motoman")
package_dirs.append("../../../motoman/robotiq")

model, collision_model, visual_model = pin.buildModelsFromUrdf(robot_urdf, package_dirs=package_dirs)

# Start a new MeshCat server and client.
# Note: the server can also be started separately using the "meshcat-server" command in a terminal:
# this enables the server to remain active after the current script ends.
#
# Option open=True pens the visualizer.
# Note: the visualizer can also be opened seperately by visiting the provided URL.    
viz = MeshcatVisualizer(model, collision_model, visual_model)
viz.initViewer(open=True)

# Load the robot in the viewer.
viz.loadViewerModel()

# Display a robot configuration.
q0 = pin.neutral(model)
print('q0: ')
print(q0)
viz.display(q0)


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7056/static/
q0: 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [6]:
import numpy as np
import time

In [7]:
model, collision_model, visual_model = pin.buildModelsFromUrdf(robot_urdf, package_dirs=package_dirs)

# *** add collisions of point cloud in FCL and then to pinocchio ***
"""
shelf_bottom:
- position: [0.85, 0, 0.5]
- size: 0.175, 0.5, 0.5
shelf_top:
- position: [0.85, 0, 1.42]
- size: 0.175, 0.5, 0.025
shelf_padding_left:
- position: [0.85, -0.475, 1.2]
- size: 0.175, 0.025, 0.2
shelf_padding_right:
- position: [0.85, 0.475, 1.2]
- size: 0.175, 0.025, 0.2
shelf_padding_back:
- position: [1.0, 0, 1.2]
- size: 0.025, 0.5, 0.2
"""
pcd_total = []
# shelf-bottom
num_points = 5000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 5000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 5000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 5000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 5000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


# add pcd to fcl
fcl_octree = hppfcl.makeOctree(pcd_total, 0.01)
# add fcl_pcd to pinocchio
octree_obj = pin.GeometryObject('octree', 0, fcl_octree, pin.SE3.Identity())
octree_obj.meshColor[0] = 1.0
collision_model.addGeometryObject(octree_obj)
# visual_model.addGeometryObject(octree_obj)

# add point cloud for visualization
point_cloud = hppfcl.BVHModelOBBRSS()
point_cloud.beginModel(0, len(pcd_total))
point_cloud.addVertices(pcd_total)
bvh_obj = pin.GeometryObject('bvh', 0, point_cloud, pin.SE3.Identity())
bvh_obj.meshColor[0] = 1.0
visual_model.addGeometryObject(bvh_obj)


# set up the collision
panda_hand_collision_id = collision_model.getGeometryId("panda_hand_0")

box_placement = pin.SE3(pin.Quaternion(1, 0, 0, 0), np.array([0,0,0]))

box = pin.GeometryObject(
    "box",
    0,  # parent joint ID (0 for world frame)
    hppfcl.Box(5, 5, 5),  # Box dimensions
    box_placement
)

box.meshColor = np.array([0, 1, 0, 1])

# collision_model.addGeometryObject(box)
# visual_model.addGeometryObject(box)

viz = MeshcatVisualizer(model, collision_model, visual_model)
viz.initViewer(open=True)

# Load the robot in the viewer.
viz.loadViewerModel()

# Display a robot configuration.
q0 = pin.neutral(model)
print('q0: ')
print(q0)
time.sleep(1.0)
viz.display(q0)
# viz.displayVisuals(True)



You can open the visualizer by visiting the following URL:
http://127.0.0.1:7057/static/


/home/kchen/Robotics/venv/lib/python3.11/site-packages/cmeel.prefix/lib/python3.11/site-packages/pinocchio/visualize/meshcat_visualizer.py:493: UserWarning: The geometry object named octree is not supported by Pinocchio/MeshCat for vizualization.
  self.loadViewerGeometryObject(collision, pin.GeometryType.COLLISION, color)


q0: 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [14]:
pcd_total.shape

(25000, 3)

In [15]:
fcl_octree = hppfcl.makeOctree(np.empty((0, 3)), 0.01)
# add fcl_pcd to pinocchio
octree_obj = pin.GeometryObject('octree', 0, fcl_octree, pin.SE3.Identity())
octree_obj.meshColor[0] = 1.0

In [8]:
viz.loadViewerGeometryObject(bvh_obj)

TypeError: MeshcatVisualizer.loadViewerGeometryObject() missing 1 required positional argument: 'geometry_type'

In [ ]:
dir(model)

['__class__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getinitargs__',
 '__getstate__',
 '__getstate_manages_dict__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__safe_for_unpickling__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'addBodyFrame',
 'addFrame',
 'addJoint',
 'addJointFrame',
 'appendBodyToJoint',
 'check',
 'copy',
 'createData',
 'damping',
 'effortLimit',
 'existBodyName',
 'existFrame',
 'existJointName',
 'frames',
 'friction',
 'getBodyId',
 'getFrameId',
 'getJointId',
 'gravity',
 'hasConfigurationLimit',
 'hasConfigurationLimitInTangent',
 'idx_qs',
 'idx_vs',
 'inertias',
 'jointPlacements',
 'joints',
 'loadFromBinary',
 'loadFromString',
 'loadFromText',
 'loadFromXML',
 'lowerPositi

: 

: 

: 

In [ ]:
for a in collision_model.geometryObjects:
    print(a.name)


torso_base_link_0
torso_link_b1_0
arm_left_link_1_s_0
arm_left_link_2_l_0
arm_left_link_3_e_0
arm_left_link_4_u_0
arm_left_link_5_r_0
arm_left_link_6_b_0
arm_left_link_7_t_0
arm_right_link_1_s_0
arm_right_link_2_l_0
arm_right_link_3_e_0
arm_right_link_4_u_0
arm_right_link_5_r_0
arm_right_link_6_b_0
arm_right_link_7_t_0
robotiq_arg2f_extra_link_0
robotiq_arg2f_base_link_0
left_outer_knuckle_0
left_outer_finger_0
left_inner_finger_0
left_inner_finger_pad_0
left_inner_knuckle_0
right_inner_knuckle_0
right_outer_knuckle_0
right_outer_finger_0
right_inner_finger_0
right_inner_finger_pad_0
camera_arm_link_0
camera_torso_link_0


In [ ]:
# if i had to guess, the difference in # of joints and q is that there are 2 gripper joints that are represented with one configuration dimension
print("joints", model.njoints)
print("q", model.nq)

joints 23
q 22


In [ ]:
for i, name in enumerate(model.names):
    print(f"joint {i}")
    print(f"name: {name}")
    # print(f"geom id: {collision_model.getGeometryId(name)}")
    print()

joint 0
name: universe

joint 1
name: torso_joint_b1

joint 2
name: arm_left_joint_1_s

joint 3
name: arm_left_joint_2_l

joint 4
name: arm_left_joint_3_e

joint 5
name: arm_left_joint_4_u

joint 6
name: arm_left_joint_5_r

joint 7
name: arm_left_joint_6_b

joint 8
name: arm_left_joint_7_t

joint 9
name: arm_right_joint_1_s

joint 10
name: arm_right_joint_2_l

joint 11
name: arm_right_joint_3_e

joint 12
name: arm_right_joint_4_u

joint 13
name: arm_right_joint_5_r

joint 14
name: arm_right_joint_6_b

joint 15
name: arm_right_joint_7_t

joint 16
name: finger_joint

joint 17
name: left_inner_finger_joint

joint 18
name: left_inner_knuckle_joint

joint 19
name: right_inner_knuckle_joint

joint 20
name: right_outer_knuckle_joint

joint 21
name: right_inner_finger_joint

joint 22
name: torso_joint_b2



In [ ]:
collision_model.__dir__()

['__module__',
 '__doc__',
 '__reduce__',
 '__init__',
 'ngeoms',
 'geometryObjects',
 'addGeometryObject',
 'removeGeometryObject',
 'getGeometryId',
 'existGeometryName',
 'createData',
 'collisionPairs',
 'addCollisionPair',
 'addAllCollisionPairs',
 'setCollisionPairs',
 'removeCollisionPair',
 'removeAllCollisionPairs',
 'existCollisionPair',
 'findCollisionPair',
 '__eq__',
 '__ne__',
 '__str__',
 '__repr__',
 'copy',
 '__copy__',
 '__deepcopy__',
 '__new__',
 '__weakref__',
 '__dict__',
 '__hash__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [ ]:
for i, geom in enumerate(collision_model.geometryObjects):
    print(i, geom.name)

print(geom.__dir__())

0 torso_base_link_0
1 torso_link_b1_0
2 arm_left_link_1_s_0
3 arm_left_link_2_l_0
4 arm_left_link_3_e_0
5 arm_left_link_4_u_0
6 arm_left_link_5_r_0
7 arm_left_link_6_b_0
8 arm_left_link_7_t_0
9 arm_right_link_1_s_0
10 arm_right_link_2_l_0
11 arm_right_link_3_e_0
12 arm_right_link_4_u_0
13 arm_right_link_5_r_0
14 arm_right_link_6_b_0
15 arm_right_link_7_t_0
16 robotiq_arg2f_extra_link_0
17 robotiq_arg2f_base_link_0
18 left_outer_knuckle_0
19 left_outer_finger_0
20 left_inner_finger_0
21 left_inner_finger_pad_0
22 left_inner_knuckle_0
23 right_inner_knuckle_0
24 right_outer_knuckle_0
25 right_outer_finger_0
26 right_inner_finger_0
27 right_inner_finger_pad_0
28 camera_arm_link_0
29 camera_torso_link_0
['__module__', '__doc__', '__reduce__', '__init__', 'meshScale', 'meshColor', 'geometry', 'name', 'parentJoint', 'parentFrame', 'placement', 'meshPath', 'overrideMaterial', 'meshTexturePath', 'disableCollision', 'meshMaterial', '__eq__', '__ne__', 'CreateCapsule', '__new__', '__weakref__', 

In [12]:
collision_model.ngeoms

31

In [10]:
# Add collisition pairs
collision_model.addAllCollisionPairs()
print("num collision pairs - initial:", len(collision_model.collisionPairs))

num collision pairs - initial: 455


In [11]:
srdf_path = '../../../motoman/motoman_sda10f_moveit_config/config/motoman_sda10f.srdf'
pin.removeCollisionPairs(model, collision_model, srdf_path)
print("num collision pairs - after:", len(collision_model.collisionPairs))

num collision pairs - after: 350


In [ ]:
print(len(collision_model.collisionPairs))
for i in range(len(collision_model.collisionPairs)):
    print('collision pair ', i, '...')
    print(collision_model.collisionPairs[i])
    print(collision_model.collisionPairs[i])

321
collision pair  0 ...
collision pair (0,2)

collision pair (0,2)

collision pair  1 ...
collision pair (0,3)

collision pair (0,3)

collision pair  2 ...
collision pair (0,4)

collision pair (0,4)

collision pair  3 ...
collision pair (0,5)

collision pair (0,5)

collision pair  4 ...
collision pair (0,6)

collision pair (0,6)

collision pair  5 ...
collision pair (0,7)

collision pair (0,7)

collision pair  6 ...
collision pair (0,8)

collision pair (0,8)

collision pair  7 ...
collision pair (0,9)

collision pair (0,9)

collision pair  8 ...
collision pair (0,10)

collision pair (0,10)

collision pair  9 ...
collision pair (0,11)

collision pair (0,11)

collision pair  10 ...
collision pair (0,12)

collision pair (0,12)

collision pair  11 ...
collision pair (0,13)

collision pair (0,13)

collision pair  12 ...
collision pair (0,14)

collision pair (0,14)

collision pair  13 ...
collision pair (0,15)

collision pair (0,15)

collision pair  14 ...
collision pair (0,16)

collision 

In [ ]:
help(pin.computeCollisions)

Help on built-in function computeCollisions in module pinocchio.pinocchio_pywrap:

computeCollisions(...)
    computeCollisions( (GeometryModel)geometry_model, (GeometryData)geometry_data, (bool)stop_at_first_collision) -> bool :
        Determine if collision pairs are effectively in collision.
    
    computeCollisions( (Model)model, (Data)data, (GeometryModel)geometry_model, (GeometryData)geometry_data, (numpy.ndarray)q, (bool)stop_at_first_collision) -> bool :
        Update the geometry for a given configuration and determine if all collision pairs are effectively in collision or not.



In [ ]:
help(pin.computeDistances)

Help on built-in function computeDistances in module pinocchio.pinocchio_pywrap:

computeDistances(...)
    computeDistances( (GeometryModel)geometry_model, (GeometryData)geometry_data) -> int :
        Compute the distance between each collision pair for a given GeometryModel and associated GeometryData.
    
    computeDistances( (Model)model, (Data)data, (GeometryModel)geometry_model, (GeometryData)geometry_data, (numpy.ndarray)q) -> int :
        Update the geometry for a given configuration and compute the distance between each collision pair



In [ ]:
is_collision = False
data = model.createData()
collision_data = collision_model.createData()
while not is_collision:
    q = pin.randomConfiguration(model)

    is_collision = pin.computeCollisions(model, data, collision_model, collision_data, q, True)

print("Found a configuration in collision:",q)
viz.display(q)


KeyboardInterrupt: 

In [ ]:
for i in range(collision_model.ngeoms):
    print(collision_model.geometryObjects[i].name)

torso_base_link_0
torso_link_b1_0
arm_left_link_1_s_0
arm_left_link_2_l_0
arm_left_link_3_e_0
arm_left_link_4_u_0
arm_left_link_5_r_0
arm_left_link_6_b_0
arm_left_link_7_t_0
arm_right_link_1_s_0
arm_right_link_2_l_0
arm_right_link_3_e_0
arm_right_link_4_u_0
arm_right_link_5_r_0
arm_right_link_6_b_0
arm_right_link_7_t_0
robotiq_arg2f_extra_link_0
robotiq_arg2f_base_link_0
left_outer_knuckle_0
left_outer_finger_0
left_inner_finger_0
left_inner_finger_pad_0
left_inner_knuckle_0
right_inner_knuckle_0
right_outer_knuckle_0
right_outer_finger_0
right_inner_finger_0
right_inner_finger_pad_0
camera_arm_link_0
camera_torso_link_0


In [ ]:
help(collision_data)

Help on GeometryData in module pinocchio.pinocchio_pywrap object:

class GeometryData(Boost.Python.instance)
 |  Geometry data linked to a Geometry Model and a Data struct.
 |  
 |  Method resolution order:
 |      GeometryData
 |      Boost.Python.instance
 |      builtins.object
 |  
 |  Static methods defined here:
 |  
 |  __copy__(...)
 |      __copy__( (GeometryData)self) -> GeometryData :
 |          Returns a copy of *this.
 |  
 |  __deepcopy__(...)
 |      __deepcopy__( (GeometryData)self, (dict)memo) -> GeometryData :
 |          Returns a deep copy of *this.
 |  
 |  __eq__(...)
 |      __eq__( (GeometryData)arg1, (GeometryData)arg2) -> object
 |  
 |  __init__(...)
 |      __init__( (object)self, (GeometryModel)geometry_model) -> None :
 |          Default constructor from a given GeometryModel
 |  
 |  __ne__(...)
 |      __ne__( (GeometryData)arg1, (GeometryData)arg2) -> object
 |  
 |  __reduce__ = <unnamed Boost.Python function>(...)
 |  
 |  __repr__(...)
 |      __re

In [ ]:
collision_data.__dir__()

['__module__',
 '__doc__',
 '__reduce__',
 '__init__',
 'oMg',
 'activeCollisionPairs',
 'distanceRequests',
 'distanceResults',
 'collisionRequests',
 'collisionResults',
 'radius',
 'fillInnerOuterObjectMaps',
 'activateCollisionPair',
 'setGeometryCollisionStatus',
 'setActiveCollisionPairs',
 'deactivateCollisionPair',
 'deactivateAllCollisionPairs',
 'setSecurityMargins',
 '__eq__',
 '__ne__',
 '__str__',
 '__repr__',
 'copy',
 '__copy__',
 '__deepcopy__',
 'saveToText',
 'loadFromText',
 'saveToString',
 'loadFromString',
 'saveToXML',
 'loadFromXML',
 'saveToBinary',
 'loadFromBinary',
 '__new__',
 '__weakref__',
 '__dict__',
 '__hash__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [ ]:
collision_data.distanceRequests.__dir__()

['__module__',
 '__doc__',
 '__reduce__',
 '__instance_size__',
 '__init__',
 '__len__',
 '__setitem__',
 '__delitem__',
 '__getitem__',
 '__contains__',
 '__iter__',
 'append',
 'extend',
 '__new__',
 '__weakref__',
 '__dict__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [ ]:
for a in collision_data.collisionResults:
    if a.isCollision():
        print(a.isCollision())
        break
print(dir(a))

True
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'addContact', 'cached_gjk_guess', 'cached_support_func_guess', 'clear', 'distance_lower_bound', 'getContact', 'getContacts', 'isCollision', 'numContacts', 'timings']


In [ ]:
collision_data.collisionResults[0].__dir__()

['__module__',
 '__doc__',
 '__reduce__',
 '__init__',
 'isCollision',
 'numContacts',
 'addContact',
 'clear',
 'getContact',
 'getContacts',
 'distance_lower_bound',
 'cached_gjk_guess',
 'cached_support_func_guess',
 'timings',
 '__new__',
 '__weakref__',
 '__dict__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [ ]:
pin.computeDistances(model, data, collision_model, collision_data, q)

26

In [ ]:
print(len(collision_data.collisionResults))
print(len(collision_model.collisionPairs))
print(collision_model.collisionPairs[0].first)
print(collision_model.geometryObjects[collision_model.collisionPairs[0].first].name)
print(collision_model.geometryObjects[collision_model.collisionPairs[0].second].name)
print(collision_data.collisionResults[0].isCollision())
print(collision_data.collisionResults[0].numContacts())

321
321
0
torso_base_link_0
arm_left_link_1_s_0
False
0


In [ ]:
for i in range(len(collision_data.collisionResults)):
    if collision_data.collisionResults[i].isCollision():
        print(i)
        

10


In [ ]:
print(collision_data.__dir__())


['__module__', '__doc__', '__reduce__', '__init__', 'oMg', 'activeCollisionPairs', 'distanceRequests', 'distanceResults', 'collisionRequests', 'collisionResults', 'radius', 'fillInnerOuterObjectMaps', 'activateCollisionPair', 'setGeometryCollisionStatus', 'setActiveCollisionPairs', 'deactivateCollisionPair', 'deactivateAllCollisionPairs', 'setSecurityMargins', '__eq__', '__ne__', '__str__', '__repr__', 'copy', '__copy__', '__deepcopy__', 'saveToText', 'loadFromText', 'saveToString', 'loadFromString', 'saveToXML', 'loadFromXML', 'saveToBinary', 'loadFromBinary', '__new__', '__weakref__', '__dict__', '__hash__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__gt__', '__ge__', '__reduce_ex__', '__getstate__', '__subclasshook__', '__init_subclass__', '__format__', '__sizeof__', '__dir__', '__class__']


In [ ]:
idx = 112
print(collision_model.geometryObjects[collision_model.collisionPairs[idx].first].name)
print(collision_model.geometryObjects[collision_model.collisionPairs[idx].second].name)
print(collision_data.collisionResults[idx].isCollision())
print(collision_data.collisionResults[idx].numContacts())
print(collision_data.collisionResults[idx].__dir__())
contacts = collision_data.collisionResults[idx].getContacts()
print(contacts[0])
print(contacts[0].__dir__())
print(contacts[0].o1)
print(contacts[0].o2)
print(contacts[0].o1.__dir__())
print(contacts[0].b1)
print(contacts[0].b2)
print(contacts[0].normal)
print(contacts[0].pos)
print(contacts[0].penetration_depth)
distance = collision_data.distanceResults[idx]
print(distance)
print(distance.__dir__())
print(distance.min_distance)
print(distance.normal)
print(distance.getNearestPoint1())
print(distance.getNearestPoint2())


arm_left_link_4_u_0
arm_right_link_5_r_0
False
0
['__module__', '__doc__', '__reduce__', '__init__', 'isCollision', 'numContacts', 'addContact', 'clear', 'getContact', 'getContacts', 'distance_lower_bound', 'cached_gjk_guess', 'cached_support_func_guess', 'timings', '__new__', '__weakref__', '__dict__', '__repr__', '__hash__', '__str__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__eq__', '__ne__', '__gt__', '__ge__', '__reduce_ex__', '__getstate__', '__subclasshook__', '__init_subclass__', '__format__', '__sizeof__', '__dir__', '__class__']


IndexError: Index out of range

In [ ]:

print(data.__dir__())

['__module__', '__doc__', '__reduce__', '__init__', 'a', 'oa', 'a_gf', 'oa_gf', 'v', 'ov', 'f', 'of', 'h', 'oMi', 'oMf', 'liMi', 'tau', 'nle', 'ddq', 'Ycrb', 'M', 'Minv', 'C', 'g', 'Fcrb', 'lastChild', 'nvSubtree', 'U', 'D', 'parents_fromRow', 'nvSubtree_fromRow', 'J', 'dJ', 'iMf', 'Ivx', 'vxI', 'B', 'Ag', 'dAg', 'hg', 'dhg', 'Ig', 'com', 'vcom', 'acom', 'mass', 'Jcom', 'dtau_dq', 'dtau_dv', 'ddq_dq', 'ddq_dv', 'kinetic_energy', 'potential_energy', 'lambda_c', 'impulse_c', 'dq_after', 'staticRegressor', 'jointTorqueRegressor', '__eq__', '__ne__', 'copy', '__copy__', '__deepcopy__', 'saveToText', 'loadFromText', 'saveToString', 'loadFromString', 'saveToXML', 'loadFromXML', 'saveToBinary', 'loadFromBinary', '__safe_for_unpickling__', '__getstate_manages_dict__', '__getinitargs__', '__getstate__', '__setstate__', '__new__', '__weakref__', '__dict__', '__repr__', '__hash__', '__str__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__gt__', '__ge__', '__reduce_ex__'

0

In [ ]:
collision_model.geometryObjects[0].__dir__()

['__module__',
 '__doc__',
 '__reduce__',
 '__init__',
 'meshScale',
 'meshColor',
 'geometry',
 'name',
 'parentJoint',
 'parentFrame',
 'placement',
 'meshPath',
 'overrideMaterial',
 'meshTexturePath',
 'disableCollision',
 'meshMaterial',
 '__eq__',
 '__ne__',
 'CreateCapsule',
 '__new__',
 '__weakref__',
 '__dict__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [ ]:
collision_model.geometryObjects[0].name

'torso_base_link_0'